In [1]:
import pandas as pd
import numpy as np
import json
import pickle
from sentence_transformers import SentenceTransformer
import faiss
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/home/arshad/Network-project/data/sample_with_alerts.csv',
                 low_memory=False)

print(f"✅ Loaded: {df.shape[0]:,} rows")
print(f"✅ Alert text sample:\n{df['alert_text'].iloc[0][:200]}...")

✅ Loaded: 435,878 rows
✅ Alert text sample:
Network flow targeting HTTP web traffic with ACK flags. Flow is predominantly unidirectional (mostly forward) with 2 forward and 0 backward packets. Packet rate is low at 0.3 packets per second with s...


In [2]:
import urllib.request
import os

MITRE_PATH = '/home/arshad/Network-project/data/enterprise-attack.json'

if not os.path.exists(MITRE_PATH):
    print("⏳ Downloading MITRE ATT&CK Enterprise dataset...")
    url = "https://raw.githubusercontent.com/mitre/cti/master/enterprise-attack/enterprise-attack.json"
    urllib.request.urlretrieve(url, MITRE_PATH)
    print("✅ Downloaded!")
else:
    print("✅ MITRE ATT&CK data already exists!")

# Load it
with open(MITRE_PATH, 'r') as f:
    mitre_data = json.load(f)

print(f"✅ Loaded MITRE ATT&CK data")
print(f"📊 Total objects in database: {len(mitre_data['objects'])}")

⏳ Downloading MITRE ATT&CK Enterprise dataset...
✅ Downloaded!
✅ Loaded MITRE ATT&CK data
📊 Total objects in database: 24771


In [3]:
techniques = []

for obj in mitre_data['objects']:
    # Only grab attack-patterns (techniques), skip deprecated ones
    if obj.get('type') != 'attack-pattern':
        continue
    if obj.get('revoked', False) or obj.get('x_mitre_deprecated', False):
        continue

    # Extract technique ID
    tech_id = None
    for ref in obj.get('external_references', []):
        if ref.get('source_name') == 'mitre-attack':
            tech_id = ref.get('external_id')
            break

    if not tech_id:
        continue

    # Extract tactic(s)
    tactics = [phase['phase_name'] 
                for phase in obj.get('kill_chain_phases', [])
                if phase['kill_chain_name'] == 'mitre-attack']

    techniques.append({
        'technique_id':   tech_id,
        'technique_name': obj.get('name', ''),
        'description':    obj.get('description', ''),
        'tactics':        ', '.join(tactics),
        'platforms':      ', '.join(obj.get('x_mitre_platforms', [])),
    })

print(f"✅ Extracted {len(techniques)} active MITRE ATT&CK techniques")

# Preview
for t in techniques[:3]:
    print(f"\n  {t['technique_id']} — {t['technique_name']}")
    print(f"  Tactics: {t['tactics']}")
    print(f"  Description: {t['description'][:150]}...")

✅ Extracted 691 active MITRE ATT&CK techniques

  T1055.011 — Extra Window Memory Injection
  Tactics: defense-evasion, privilege-escalation
  Description: Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate ...

  T1053.005 — Scheduled Task
  Tactics: execution, persistence, privilege-escalation
  Description: Adversaries may abuse the Windows Task Scheduler to perform task scheduling for initial or recurring execution of malicious code. There are multiple w...

  T1205.002 — Socket Filters
  Tactics: defense-evasion, persistence, command-and-control
  Description: Adversaries may attach filters to a network socket to monitor then activate backdoors used for persistence or command and control. With elevated permi...


In [4]:
def build_technique_text(t):
    """
    Creates a rich text representation of each MITRE technique
    for embedding. More context = better matching.
    """
    return (
        f"Technique {t['technique_id']}: {t['technique_name']}. "
        f"Tactics: {t['tactics']}. "
        f"Platforms: {t['platforms']}. "
        f"Description: {t['description'][:500]}"
    )

technique_texts = [build_technique_text(t) for t in techniques]

print(f"✅ Built {len(technique_texts)} technique texts")
print(f"\n📝 Example technique text:")
print("-" * 60)
print(technique_texts[0][:400])

✅ Built 691 technique texts

📝 Example technique text:
------------------------------------------------------------
Technique T1055.011: Extra Window Memory Injection. Tactics: defense-evasion, privilege-escalation. Platforms: Windows. Description: Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 




In [5]:
print("⏳ Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded!")

print(f"\n⏳ Embedding {len(technique_texts)} MITRE techniques...")
print("   (This only runs once — we'll save and reuse these embeddings)")

technique_embeddings = model.encode(
    technique_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ Technique embeddings shape: {technique_embeddings.shape}")
print(f"   → {technique_embeddings.shape[0]} techniques")
print(f"   → {technique_embeddings.shape[1]} dimensions per embedding")

⏳ Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded!

⏳ Embedding 691 MITRE techniques...
   (This only runs once — we'll save and reuse these embeddings)


Batches:   0%|          | 0/11 [00:00<?, ?it/s]


✅ Technique embeddings shape: (691, 384)
   → 691 techniques
   → 384 dimensions per embedding


In [6]:
# Normalize embeddings for cosine similarity
faiss.normalize_L2(technique_embeddings)

# Build the index
embedding_dim = technique_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # Inner Product = cosine similarity after normalization
index.add(technique_embeddings)

print(f"✅ FAISS index built!")
print(f"   → {index.ntotal} techniques indexed")
print(f"   → Ready for similarity search")

✅ FAISS index built!
   → 691 techniques indexed
   → Ready for similarity search


In [7]:
SAVE_DIR = '/home/arshad/Network-project/data/'

# Save FAISS index
faiss.write_index(index, SAVE_DIR + 'mitre_faiss.index')

# Save technique metadata
with open(SAVE_DIR + 'mitre_techniques.pkl', 'wb') as f:
    pickle.dump(techniques, f)

# Save technique embeddings
np.save(SAVE_DIR + 'mitre_embeddings.npy', technique_embeddings)

print("✅ Saved:")
print(f"   → mitre_faiss.index")
print(f"   → mitre_techniques.pkl")
print(f"   → mitre_embeddings.npy")

✅ Saved:
   → mitre_faiss.index
   → mitre_techniques.pkl
   → mitre_embeddings.npy


In [8]:
def match_to_mitre(alert_text, model, index, techniques, top_k=3):
    """
    Takes an alert text and returns the top-k most similar
    MITRE ATT&CK techniques with confidence scores.
    """
    # Embed the alert
    alert_embedding = model.encode([alert_text], convert_to_numpy=True)
    faiss.normalize_L2(alert_embedding)

    # Search FAISS index
    similarities, indices = index.search(alert_embedding, top_k)

    # Build results
    results = []
    for sim, idx in zip(similarities[0], indices[0]):
        t = techniques[idx]
        results.append({
            'technique_id':   t['technique_id'],
            'technique_name': t['technique_name'],
            'tactics':        t['tactics'],
            'confidence':     round(float(sim), 4)
        })

    return results

print("✅ Matching function defined!")

✅ Matching function defined!


In [9]:
print("🧪 TESTING MITRE MATCHING ON REAL ALERTS")
print("=" * 70)

# Test one sample from each major attack type
test_labels = ['DDoS', 'PortScan', 'Bot', 'FTP-Patator', 
               'DoS Hulk', 'Web Attack - SQL Injection']

for label in test_labels:
    sample_row = df[df['Label'] == label].iloc[0]
    alert_text  = sample_row['alert_text']
    true_technique = sample_row['MITRE_Technique']
    true_tactic    = sample_row['MITRE_Tactic']

    matches = match_to_mitre(alert_text, model, index, techniques, top_k=3)

    print(f"\n🏷️  Attack: {label}")
    print(f"   ✅ True MITRE: {true_technique} ({true_tactic})")
    print(f"   🔍 Top-3 Predicted Matches:")
    for i, m in enumerate(matches, 1):
        hit = "🎯" if true_technique in m['technique_id'] else "  "
        print(f"   {hit} {i}. {m['technique_id']} — {m['technique_name']}")
        print(f"         Tactics: {m['tactics']}")
        print(f"         Confidence: {m['confidence']:.4f}")

🧪 TESTING MITRE MATCHING ON REAL ALERTS

🏷️  Attack: DDoS
   ✅ True MITRE: T1498 (TA0040)
   🔍 Top-3 Predicted Matches:
      1. T1071.001 — Web Protocols
         Tactics: command-and-control
         Confidence: 0.4769
   🎯 2. T1498.001 — Direct Network Flood
         Tactics: impact
         Confidence: 0.4015
      3. T1499.002 — Service Exhaustion Flood
         Tactics: impact
         Confidence: 0.3815

🏷️  Attack: PortScan
   ✅ True MITRE: T1046 (TA0007)
   🔍 Top-3 Predicted Matches:
      1. T1498.001 — Direct Network Flood
         Tactics: impact
         Confidence: 0.4057
      2. T1205.002 — Socket Filters
         Tactics: defense-evasion, persistence, command-and-control
         Confidence: 0.3785
      3. T1568.003 — DNS Calculation
         Tactics: command-and-control
         Confidence: 0.3672

🏷️  Attack: Bot
   ✅ True MITRE: T1071 (TA0011)
   🔍 Top-3 Predicted Matches:
   🎯 1. T1071.001 — Web Protocols
         Tactics: command-and-control
         Confidence: 

In [10]:
print("⏳ Applying MITRE matching to all 435,878 alerts...")
print("   This will take a few minutes...\n")

# Batch embed all alert texts for efficiency
alert_texts = df['alert_text'].tolist()

alert_embeddings = model.encode(
    alert_texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Normalize for cosine similarity
faiss.normalize_L2(alert_embeddings)

print(f"\n✅ Alert embeddings shape: {alert_embeddings.shape}")

# Save alert embeddings too
np.save(SAVE_DIR + 'alert_embeddings.npy', alert_embeddings)
print("✅ Alert embeddings saved!")

⏳ Applying MITRE matching to all 435,878 alerts...
   This will take a few minutes...



Batches:   0%|          | 0/1703 [00:00<?, ?it/s]


✅ Alert embeddings shape: (435878, 384)
✅ Alert embeddings saved!


In [11]:
print("⏳ Searching FAISS index for all alerts...")

# Search all at once — FAISS handles this very efficiently
similarities, indices = index.search(alert_embeddings, 3)  # top 3

# Add results to dataframe
for rank in range(3):
    df[f'pred_technique_{rank+1}']   = [techniques[i]['technique_id']   for i in indices[:, rank]]
    df[f'pred_tech_name_{rank+1}']   = [techniques[i]['technique_name'] for i in indices[:, rank]]
    df[f'pred_tactic_{rank+1}']      = [techniques[i]['tactics']        for i in indices[:, rank]]
    df[f'pred_confidence_{rank+1}']  = similarities[:, rank].round(4)

print("✅ Top-3 MITRE matches added to dataframe!")
print(f"\n📊 New columns added:")
new_cols = [c for c in df.columns if 'pred_' in c]
print(df[['Label', 'MITRE_Technique'] + new_cols].iloc[0])

⏳ Searching FAISS index for all alerts...
✅ Top-3 MITRE matches added to dataframe!

📊 New columns added:
Label                                                    DoS GoldenEye
MITRE_Technique                                                  T1499
pred_technique_1                                             T1071.001
pred_tech_name_1                                         Web Protocols
pred_tactic_1                                      command-and-control
pred_confidence_1                                               0.5245
pred_technique_2                                             T1499.002
pred_tech_name_2                              Service Exhaustion Flood
pred_tactic_2                                                   impact
pred_confidence_2                                               0.4529
pred_technique_3                                             T1205.002
pred_tech_name_3                                        Socket Filters
pred_tactic_3        defense-evasion, pers

In [12]:
# Check if true technique appears in top-3 predictions (for non-BENIGN)
attack_df = df[df['Label'] != 'BENIGN'].copy()

# Top-1 accuracy
top1 = (attack_df['MITRE_Technique'] == attack_df['pred_technique_1']).mean()

# Top-3 accuracy (true technique in any of top 3)
top3 = attack_df.apply(
    lambda row: row['MITRE_Technique'] in [
        row['pred_technique_1'],
        row['pred_technique_2'],
        row['pred_technique_3']
    ], axis=1
).mean()

print("📊 MITRE MATCHING ACCURACY")
print("=" * 40)
print(f"   Top-1 Accuracy: {top1:.2%}")
print(f"   Top-3 Accuracy: {top3:.2%}")
print()
print("Note: Zero-shot embedding matching — no training required!")
print("This is expected to be moderate; risk scoring will refine results.")

📊 MITRE MATCHING ACCURACY
   Top-1 Accuracy: 0.00%
   Top-3 Accuracy: 0.01%

Note: Zero-shot embedding matching — no training required!
This is expected to be moderate; risk scoring will refine results.


In [13]:
OUTPUT_PATH = '/home/arshad/Network-project/data/sample_with_mitre_matches.csv'
df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Saved final dataset with MITRE matches!")
print(f"📊 Shape: {df.shape}")
print(f"📁 Location: {OUTPUT_PATH}")

✅ Saved final dataset with MITRE matches!
📊 Shape: (435878, 49)
📁 Location: /home/arshad/Network-project/data/sample_with_mitre_matches.csv


In [14]:
def get_parent_id(technique_id):
    """T1499.002 → T1499,  T1071.001 → T1071,  T1499 → T1499"""
    return technique_id.split('.')[0] if '.' in technique_id else technique_id

attack_df = df[df['Label'] != 'BENIGN'].copy()

# Normalize all IDs to parent level
attack_df['true_parent']  = attack_df['MITRE_Technique'].apply(get_parent_id)
attack_df['pred1_parent'] = attack_df['pred_technique_1'].apply(get_parent_id)
attack_df['pred2_parent'] = attack_df['pred_technique_2'].apply(get_parent_id)
attack_df['pred3_parent'] = attack_df['pred_technique_3'].apply(get_parent_id)

# Top-1 accuracy
top1 = (attack_df['true_parent'] == attack_df['pred1_parent']).mean()

# Top-3 accuracy
top3 = attack_df.apply(
    lambda row: row['true_parent'] in [
        row['pred1_parent'],
        row['pred2_parent'],
        row['pred3_parent']
    ], axis=1
).mean()

# Per-attack-type breakdown
print("📊 CORRECTED MITRE MATCHING ACCURACY")
print("=" * 50)
print(f"   Top-1 Accuracy: {top1:.2%}")
print(f"   Top-3 Accuracy: {top3:.2%}")

print("\n📊 Per Attack Type Breakdown:")
print("-" * 50)
for label in attack_df['Label'].unique():
    subset = attack_df[attack_df['Label'] == label]
    t1 = (subset['true_parent'] == subset['pred1_parent']).mean()
    t3 = subset.apply(
        lambda row: row['true_parent'] in [
            row['pred1_parent'], 
            row['pred2_parent'], 
            row['pred3_parent']
        ], axis=1
    ).mean()
    print(f"   {label:<35} Top-1: {t1:.0%}  Top-3: {t3:.0%}")

📊 CORRECTED MITRE MATCHING ACCURACY
   Top-1 Accuracy: 0.29%
   Top-3 Accuracy: 66.02%

📊 Per Attack Type Breakdown:
--------------------------------------------------
   DoS GoldenEye                       Top-1: 0%  Top-3: 99%
   PortScan                            Top-1: 0%  Top-3: 0%
   DoS Hulk                            Top-1: 0%  Top-3: 100%
   DDoS                                Top-1: 0%  Top-3: 67%
   DoS Slowhttptest                    Top-1: 0%  Top-3: 100%
   SSH-Patator                         Top-1: 0%  Top-3: 0%
   DoS slowloris                       Top-1: 0%  Top-3: 100%
   FTP-Patator                         Top-1: 0%  Top-3: 0%
   Bot                                 Top-1: 64%  Top-3: 68%
   Web Attack - Brute Force            Top-1: 0%  Top-3: 0%
   Web Attack - XSS                    Top-1: 0%  Top-3: 0%
   Heartbleed                          Top-1: 0%  Top-3: 0%
   Infiltration                        Top-1: 0%  Top-3: 0%
   Web Attack - SQL Injection          Top

In [15]:
def generate_enhanced_alert_text(row):
    """
    Enhanced version with stronger semantic signals
    for harder-to-match attack types.
    """
    dst_port  = int(row.get('Destination Port', 0))
    syn       = int(row.get('SYN Flag Count', 0))
    ack       = int(row.get('ACK Flag Count', 0))
    fin       = int(row.get('FIN Flag Count', 0))
    rst       = int(row.get('RST Flag Count', 0))
    psh       = int(row.get('PSH Flag Count', 0))
    fwd_pkts  = int(row.get('Total Fwd Packets', 0))
    bwd_pkts  = int(row.get('Total Backward Packets', 0))
    pkt_rate  = float(row.get('Flow Packets/s', 0))
    pkt_len   = float(row.get('Packet Length Mean', 0))
    byte_rate = float(row.get('Flow Bytes/s', 0))
    duration  = float(row.get('Flow Duration', 0)) / 1e6
    iat_mean  = float(row.get('Flow IAT Mean', 0))
    init_win  = int(row.get('Init_Win_bytes_forward', 0))
    down_up   = float(row.get('Down/Up Ratio', 0))

    # Port context
    port_context = {
        80: 'HTTP web server',     443: 'HTTPS web server',
        22: 'SSH remote access',   21:  'FTP file transfer',
        23: 'Telnet service',      3389:'RDP remote desktop',
        53: 'DNS resolver',        3306:'MySQL database server',
        8080:'HTTP web proxy',     445: 'SMB file sharing',
        25: 'SMTP mail server',    110: 'POP3 mail',
        143:'IMAP mail',           1433:'MSSQL database'
    }
    port_desc = port_context.get(dst_port, f'uncommon port {dst_port}')

    # Behavior pattern detection
    behaviors = []

    # Port scanning pattern
    if pkt_rate > 5000 and pkt_len < 10 and fwd_pkts <= 2:
        behaviors.append(
            "rapid probing of multiple ports suggesting network reconnaissance or port scanning"
        )

    # DoS/DDoS pattern
    if pkt_rate > 500 and bwd_pkts < fwd_pkts * 0.1:
        behaviors.append(
            "high volume unidirectional flood suggesting denial of service attack"
        )

    # Brute force pattern
    if syn > 0 and duration > 1 and pkt_rate < 1000 and dst_port in [21, 22, 23, 3389]:
        behaviors.append(
            f"repeated connection attempts to {port_desc} suggesting brute force credential attack"
        )

    # C2 / Botnet pattern
    if iat_mean > 10000 and bwd_pkts > 0 and pkt_len < 200:
        behaviors.append(
            "periodic low-volume bidirectional communication suggesting command and control beaconing"
        )

    # Web attack pattern
    if dst_port in [80, 443, 8080] and psh > 0 and pkt_len > 50:
        behaviors.append(
            "HTTP payload activity targeting web application suggesting possible web-based attack"
        )

    # Data exfiltration pattern
    if down_up > 5 and byte_rate > 10000:
        behaviors.append(
            "high outbound data volume suggesting possible data exfiltration"
        )

    behavior_str = '. '.join(behaviors) if behaviors else \
                   "standard network communication pattern"

    # Build enriched alert
    flags = []
    if syn: flags.append('SYN')
    if ack: flags.append('ACK')
    if fin: flags.append('FIN')
    if rst: flags.append('RST')
    if psh: flags.append('PSH')
    flag_str = ', '.join(flags) if flags else 'no TCP flags'

    alert = (
        f"Network flow to {port_desc} with {flag_str} flags. "
        f"Traffic pattern: {behavior_str}. "
        f"Flow stats: {fwd_pkts} forward packets, {bwd_pkts} backward packets, "
        f"{pkt_rate:.1f} packets/sec, {pkt_len:.1f} bytes mean packet size, "
        f"{duration:.2f} second duration, {byte_rate:.1f} bytes/sec throughput. "
        f"Initial window size: {init_win} bytes."
    )

    return alert

# Test on the problem cases
print("🧪 Testing enhanced alert text on problem cases:\n")
for label in ['PortScan', 'Web Attack - SQL Injection', 'FTP-Patator']:
    row = df[df['Label'] == label].iloc[0]
    print(f"🏷️  {label}")
    print(f"   {generate_enhanced_alert_text(row)[:300]}")
    print()

🧪 Testing enhanced alert text on problem cases:

🏷️  PortScan
   Network flow to uncommon port 2718 with PSH flags. Traffic pattern: rapid probing of multiple ports suggesting network reconnaissance or port scanning. Flow stats: 1 forward packets, 1 backward packets, 24691.4 packets/sec, 3.3 bytes mean packet size, 0.00 second duration, 98765.4 bytes/sec throughp

🏷️  Web Attack - SQL Injection
   Network flow to HTTP web server with PSH flags. Traffic pattern: HTTP payload activity targeting web application suggesting possible web-based attack. Flow stats: 5 forward packets, 3 backward packets, 1.6 packets/sec, 268.7 bytes mean packet size, 5.01 second duration, 482.9 bytes/sec throughput. I

🏷️  FTP-Patator
   Network flow to FTP file transfer with PSH flags. Traffic pattern: periodic low-volume bidirectional communication suggesting command and control beaconing. Flow stats: 9 forward packets, 15 backward packets, 2.6 packets/sec, 11.4 bytes mean packet size, 9.16 second duration, 3

In [16]:
from tqdm import tqdm
tqdm.pandas()

print("⏳ Generating enhanced alert texts...")
df['alert_text'] = df.progress_apply(generate_enhanced_alert_text, axis=1)
print("✅ Enhanced alert texts generated!")

# Re-embed with enhanced texts
print("\n⏳ Re-embedding with enhanced alert texts...")
alert_embeddings_v2 = model.encode(
    df['alert_text'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)
faiss.normalize_L2(alert_embeddings_v2)

# Re-search
similarities2, indices2 = index.search(alert_embeddings_v2, 3)

# Update predictions
for rank in range(3):
    df[f'pred_technique_{rank+1}']  = [techniques[i]['technique_id']   for i in indices2[:, rank]]
    df[f'pred_tech_name_{rank+1}']  = [techniques[i]['technique_name'] for i in indices2[:, rank]]
    df[f'pred_tactic_{rank+1}']     = [techniques[i]['tactics']        for i in indices2[:, rank]]
    df[f'pred_confidence_{rank+1}'] = similarities2[:, rank].round(4)

print("✅ Predictions updated with enhanced embeddings!")

⏳ Generating enhanced alert texts...


100%|████████████████████████████████| 435878/435878 [00:07<00:00, 55668.91it/s]


✅ Enhanced alert texts generated!

⏳ Re-embedding with enhanced alert texts...


Batches:   0%|          | 0/1703 [00:00<?, ?it/s]

✅ Predictions updated with enhanced embeddings!


In [17]:
attack_df = df[df['Label'] != 'BENIGN'].copy()
attack_df['true_parent']  = attack_df['MITRE_Technique'].apply(get_parent_id)
attack_df['pred1_parent'] = attack_df['pred_technique_1'].apply(get_parent_id)
attack_df['pred2_parent'] = attack_df['pred_technique_2'].apply(get_parent_id)
attack_df['pred3_parent'] = attack_df['pred_technique_3'].apply(get_parent_id)

top1 = (attack_df['true_parent'] == attack_df['pred1_parent']).mean()
top3 = attack_df.apply(
    lambda row: row['true_parent'] in [
        row['pred1_parent'],
        row['pred2_parent'],
        row['pred3_parent']
    ], axis=1
).mean()

print("📊 FINAL MITRE MATCHING ACCURACY (Enhanced)")
print("=" * 50)
print(f"   Top-1 Accuracy: {top1:.2%}")
print(f"   Top-3 Accuracy: {top3:.2%}")
print()
print("📊 Per Attack Type:")
print("-" * 50)
for label in sorted(attack_df['Label'].unique()):
    subset = attack_df[attack_df['Label'] == label]
    t1 = (subset['true_parent'] == subset['pred1_parent']).mean()
    t3 = subset.apply(
        lambda row: row['true_parent'] in [
            row['pred1_parent'],
            row['pred2_parent'],
            row['pred3_parent']
        ], axis=1
    ).mean()
    status = "✅" if t3 > 0.3 else "⚠️ "
    print(f"   {status} {label:<35} Top-1: {t1:.0%}  Top-3: {t3:.0%}")

# Save final
df.to_csv('/home/arshad/Network-project/data/sample_with_mitre_matches.csv', index=False)
print("\n✅ Final dataset saved!")

📊 FINAL MITRE MATCHING ACCURACY (Enhanced)
   Top-1 Accuracy: 2.12%
   Top-3 Accuracy: 45.79%

📊 Per Attack Type:
--------------------------------------------------
   ✅ Bot                                 Top-1: 64%  Top-3: 64%
   ⚠️  DDoS                                Top-1: 0%  Top-3: 0%
   ✅ DoS GoldenEye                       Top-1: 1%  Top-3: 100%
   ✅ DoS Hulk                            Top-1: 4%  Top-3: 100%
   ✅ DoS Slowhttptest                    Top-1: 0%  Top-3: 100%
   ✅ DoS slowloris                       Top-1: 0%  Top-3: 100%
   ⚠️  FTP-Patator                         Top-1: 0%  Top-3: 0%
   ⚠️  Heartbleed                          Top-1: 0%  Top-3: 0%
   ⚠️  Infiltration                        Top-1: 0%  Top-3: 0%
   ⚠️  PortScan                            Top-1: 0%  Top-3: 0%
   ⚠️  SSH-Patator                         Top-1: 0%  Top-3: 0%
   ⚠️  Web Attack - Brute Force            Top-1: 0%  Top-3: 0%
   ⚠️  Web Attack - SQL Injection          Top-1: 0%  Top-3: 0%
   

In [18]:
print("🔍 DIAGNOSING FAILING ATTACK TYPES")
print("=" * 70)

failing_labels = ['PortScan', 'FTP-Patator', 'SSH-Patator', 
                  'Heartbleed', 'Web Attack - Brute Force']

for label in failing_labels:
    subset = df[df['Label'] == label].iloc[0]
    print(f"\n🏷️  {label}")
    print(f"   True Technique: {subset['MITRE_Technique']} ({subset['MITRE_Tech_Name']})")
    print(f"   Alert Text: {subset['alert_text'][:200]}")
    print(f"   Predicted #1: {subset['pred_technique_1']} — {subset['pred_tech_name_1']}")
    print(f"   Predicted #2: {subset['pred_technique_2']} — {subset['pred_tech_name_2']}")
    print(f"   Predicted #3: {subset['pred_technique_3']} — {subset['pred_tech_name_3']}")
    
    # Check: does the true technique even exist in our MITRE database?
    true_id = subset['MITRE_Technique']
    matches_in_db = [t for t in techniques if true_id in t['technique_id']]
    print(f"   True technique in MITRE DB: {'✅ YES' if matches_in_db else '❌ NOT FOUND'}")
    if matches_in_db:
        print(f"   DB entry: {matches_in_db[0]['technique_id']} — {matches_in_db[0]['technique_name']}")
        print(f"   DB description: {matches_in_db[0]['description'][:200]}")

🔍 DIAGNOSING FAILING ATTACK TYPES

🏷️  PortScan
   True Technique: T1046 (Network Service Discovery)
   Alert Text: Network flow to uncommon port 2718 with PSH flags. Traffic pattern: rapid probing of multiple ports suggesting network reconnaissance or port scanning. Flow stats: 1 forward packets, 1 backward packet
   Predicted #1: T1205.001 — Port Knocking
   Predicted #2: T1205.002 — Socket Filters
   Predicted #3: T1568.003 — DNS Calculation
   True technique in MITRE DB: ✅ YES
   DB entry: T1046 — Network Service Discovery
   DB description: Adversaries may attempt to get a listing of services running on remote hosts and local network infrastructure devices, including those that may be vulnerable to remote software exploitation. Common me

🏷️  FTP-Patator
   True Technique: T1110 (Brute Force)
   Alert Text: Network flow to FTP file transfer with PSH flags. Traffic pattern: periodic low-volume bidirectional communication suggesting command and control beaconing. Flow stats: 9 forwa

In [19]:
def generate_targeted_alert_text(row):
    """
    Generates semantically rich alert text by combining
    network stats WITH attack-specific behavioral language.
    Uses port + flag + traffic patterns to pick the right template.
    """
    dst_port  = int(row.get('Destination Port', 0))
    syn       = int(row.get('SYN Flag Count', 0))
    ack       = int(row.get('ACK Flag Count', 0))
    fin       = int(row.get('FIN Flag Count', 0))
    rst       = int(row.get('RST Flag Count', 0))
    psh       = int(row.get('PSH Flag Count', 0))
    fwd_pkts  = int(row.get('Total Fwd Packets', 0))
    bwd_pkts  = int(row.get('Total Backward Packets', 0))
    pkt_rate  = float(row.get('Flow Packets/s', 0))
    pkt_len   = float(row.get('Packet Length Mean', 0))
    byte_rate = float(row.get('Flow Bytes/s', 0))
    duration  = float(row.get('Flow Duration', 0)) / 1e6
    iat_mean  = float(row.get('Flow IAT Mean', 0))
    init_win  = int(row.get('Init_Win_bytes_forward', 0))
    down_up   = float(row.get('Down/Up Ratio', 0))
    flow_pkts = fwd_pkts + bwd_pkts

    # Port descriptions
    port_context = {
        80: 'HTTP web server',      443: 'HTTPS web server',
        22: 'SSH remote access',    21:  'FTP file transfer server',
        23: 'Telnet service',       3389:'RDP remote desktop',
        53: 'DNS resolver',         3306:'MySQL database server',
        8080:'HTTP web proxy',      445: 'SMB file sharing',
        25: 'SMTP mail server',     110: 'POP3 mail server',
        143:'IMAP mail server',     1433:'MSSQL database server'
    }
    port_desc = port_context.get(dst_port, f'port {dst_port}')

    flags = []
    if syn: flags.append('SYN')
    if ack: flags.append('ACK')
    if fin: flags.append('FIN')
    if rst: flags.append('RST')
    if psh: flags.append('PSH')
    flag_str = ', '.join(flags) if flags else 'no TCP flags'

    # ── Pattern 1: Port Scanning ──────────────────────────────────────
    # Very small packets, very few packets per flow, high packet rate
    if pkt_len < 20 and flow_pkts <= 3 and pkt_rate > 1000:
        return (
            f"Network reconnaissance activity scanning {port_desc}. "
            f"Adversary is enumerating open network services and ports "
            f"to discover vulnerable remote hosts. "
            f"Tiny {pkt_len:.1f} byte probes sent at {pkt_rate:.0f} packets/sec "
            f"with {flag_str} flags. Flow duration {duration:.3f} seconds. "
            f"Typical of automated network service discovery and port scanning tools."
        )

    # ── Pattern 2: Brute Force on auth services ───────────────────────
    # Auth port, repeated short bidirectional flows, moderate rate
    if dst_port in [21, 22, 23, 3389, 5900] and bwd_pkts > 0 and duration > 0.5:
        service = {21:'FTP', 22:'SSH', 23:'Telnet', 
                   3389:'RDP', 5900:'VNC'}.get(dst_port, 'remote service')
        return (
            f"Repeated {service} authentication attempts targeting {port_desc}. "
            f"Adversary is systematically guessing passwords and credentials "
            f"to gain unauthorized access via brute force attack. "
            f"{fwd_pkts} forward and {bwd_pkts} backward packets at "
            f"{pkt_rate:.1f} packets/sec with {flag_str} flags. "
            f"Multiple login attempts over {duration:.2f} seconds. "
            f"Consistent with credential brute forcing and password guessing attacks."
        )

    # ── Pattern 3: DoS / Flood attacks ───────────────────────────────
    # High rate, mostly forward, large or small packets
    if pkt_rate > 500 and fwd_pkts > bwd_pkts * 3:
        return (
            f"High volume denial of service flood attack targeting {port_desc}. "
            f"Adversary sending {pkt_rate:.0f} packets/sec to exhaust server resources "
            f"and deny legitimate users access to the service. "
            f"{fwd_pkts} forward vs {bwd_pkts} backward packets, "
            f"mean packet size {pkt_len:.1f} bytes, {flag_str} flags. "
            f"Byte rate {byte_rate:.0f} bytes/sec over {duration:.2f} seconds. "
            f"Consistent with endpoint denial of service and network flood attacks."
        )

    # ── Pattern 4: DDoS ───────────────────────────────────────────────
    if pkt_rate > 100 and pkt_len > 500 and bwd_pkts < fwd_pkts:
        return (
            f"Distributed denial of service attack targeting {port_desc}. "
            f"Large volume of {pkt_len:.0f} byte packets flooding the network "
            f"at {pkt_rate:.0f} packets/sec to overwhelm target infrastructure. "
            f"Network layer flood with {flag_str} flags consuming {byte_rate:.0f} bytes/sec bandwidth. "
            f"Consistent with network denial of service and distributed flood attack."
        )

    # ── Pattern 5: Web application attacks ────────────────────────────
    if dst_port in [80, 443, 8080] and psh > 0 and flow_pkts > 2:
        return (
            f"Web application attack targeting {port_desc}. "
            f"Adversary exploiting vulnerabilities in public-facing web application "
            f"through malicious HTTP requests including possible SQL injection, "
            f"cross-site scripting XSS, or brute force login attempts. "
            f"{fwd_pkts} request packets with {pkt_len:.1f} byte payloads "
            f"over {duration:.2f} seconds. "
            f"Consistent with exploitation of public-facing web application vulnerabilities."
        )

    # ── Pattern 6: C2 Botnet beaconing ───────────────────────────────
    if iat_mean > 50000 and bwd_pkts > 0 and pkt_len < 300:
        return (
            f"Command and control communication from compromised host via {port_desc}. "
            f"Bot malware beaconing to remote C2 server using application layer protocol. "
            f"Periodic {pkt_len:.1f} byte packets with {iat_mean:.0f} microsecond "
            f"inter-arrival time and {flag_str} flags. "
            f"Bidirectional flow: {fwd_pkts} sent, {bwd_pkts} received. "
            f"Consistent with botnet command and control channel communication."
        )

    # ── Pattern 7: Exploitation / Vulnerability attacks ───────────────
    if pkt_len > 800 and duration > 10 and flow_pkts > 100:
        return (
            f"Possible exploitation of remote service vulnerability at {port_desc}. "
            f"Large {pkt_len:.0f} byte payloads over extended {duration:.1f} second session "
            f"suggesting adversary exploiting software vulnerability for unauthorized access. "
            f"{fwd_pkts} forward and {bwd_pkts} backward packets with {flag_str} flags. "
            f"Consistent with exploitation of remote services and vulnerability exploitation."
        )

    # ── Pattern 8: Data exfiltration ──────────────────────────────────
    if down_up > 5 and byte_rate > 5000:
        return (
            f"Suspicious outbound data transfer from {port_desc}. "
            f"High volume of data leaving network at {byte_rate:.0f} bytes/sec "
            f"with down/up ratio of {down_up:.1f} suggesting data exfiltration. "
            f"{fwd_pkts} forward and {bwd_pkts} backward packets over {duration:.2f} seconds. "
            f"Consistent with automated data exfiltration and lateral tool transfer."
        )

    # ── Default: Generic flow ──────────────────────────────────────────
    return (
        f"Network flow targeting {port_desc} with {flag_str} flags. "
        f"{fwd_pkts} forward and {bwd_pkts} backward packets "
        f"at {pkt_rate:.1f} packets/sec with {pkt_len:.1f} byte mean packet size. "
        f"Flow duration {duration:.2f} seconds at {byte_rate:.1f} bytes/sec."
    )

print("✅ Targeted alert text generator defined!")

# Preview all attack types
print("\n📝 Sample alerts per attack type:")
print("=" * 70)
for label in df['Label'].unique():
    row = df[df['Label'] == label].iloc[0]
    text = generate_targeted_alert_text(row)
    print(f"\n🏷️  {label}")
    print(f"   {text[:250]}")

✅ Targeted alert text generator defined!

📝 Sample alerts per attack type:

🏷️  DoS GoldenEye
   Network flow targeting HTTP web server with ACK flags. 2 forward and 0 backward packets at 0.3 packets/sec with 0.0 byte mean packet size. Flow duration 7.66 seconds at 0.0 bytes/sec.

🏷️  PortScan
   Network reconnaissance activity scanning port 2718. Adversary is enumerating open network services and ports to discover vulnerable remote hosts. Tiny 3.3 byte probes sent at 24691 packets/sec with PSH flags. Flow duration 0.000 seconds. Typical of a

🏷️  DoS Hulk
   Network flow targeting HTTP web server with FIN flags. 6 forward and 7 backward packets at 0.2 packets/sec with 854.3 byte mean packet size. Flow duration 83.82 seconds at 142.6 bytes/sec.

🏷️  DDoS
   Web application attack targeting HTTP web server. Adversary exploiting vulnerabilities in public-facing web application through malicious HTTP requests including possible SQL injection, cross-site scripting XSS, or brute force login

In [20]:
from tqdm import tqdm
tqdm.pandas()

# Generate new alert texts
print("⏳ Generating targeted alert texts...")
df['alert_text'] = df.progress_apply(generate_targeted_alert_text, axis=1)
print("✅ Done!")

# Re-embed
print("\n⏳ Re-embedding all alerts...")
alert_embeddings_v3 = model.encode(
    df['alert_text'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)
faiss.normalize_L2(alert_embeddings_v3)

# Re-search
similarities3, indices3 = index.search(alert_embeddings_v3, 3)

for rank in range(3):
    df[f'pred_technique_{rank+1}']  = [techniques[i]['technique_id']   for i in indices3[:, rank]]
    df[f'pred_tech_name_{rank+1}']  = [techniques[i]['technique_name'] for i in indices3[:, rank]]
    df[f'pred_tactic_{rank+1}']     = [techniques[i]['tactics']        for i in indices3[:, rank]]
    df[f'pred_confidence_{rank+1}'] = similarities3[:, rank].round(4)

print("✅ Predictions updated!")

⏳ Generating targeted alert texts...


100%|████████████████████████████████| 435878/435878 [00:07<00:00, 56729.60it/s]


✅ Done!

⏳ Re-embedding all alerts...


Batches:   0%|          | 0/1703 [00:00<?, ?it/s]

✅ Predictions updated!


In [21]:
attack_df = df[df['Label'] != 'BENIGN'].copy()
attack_df['true_parent']  = attack_df['MITRE_Technique'].apply(get_parent_id)
attack_df['pred1_parent'] = attack_df['pred_technique_1'].apply(get_parent_id)
attack_df['pred2_parent'] = attack_df['pred_technique_2'].apply(get_parent_id)
attack_df['pred3_parent'] = attack_df['pred_technique_3'].apply(get_parent_id)

top1 = (attack_df['true_parent'] == attack_df['pred1_parent']).mean()
top3 = attack_df.apply(
    lambda row: row['true_parent'] in [
        row['pred1_parent'],
        row['pred2_parent'],
        row['pred3_parent']
    ], axis=1
).mean()

print("📊 FINAL ACCURACY — Targeted Alert Text")
print("=" * 55)
print(f"   Top-1 Accuracy: {top1:.2%}")
print(f"   Top-3 Accuracy: {top3:.2%}")
print()
print(f"{'Attack Type':<35} {'Top-1':>6}  {'Top-3':>6}  {'Count':>7}")
print("-" * 58)
for label in sorted(attack_df['Label'].unique()):
    subset = attack_df[attack_df['Label'] == label]
    t1 = (subset['true_parent'] == subset['pred1_parent']).mean()
    t3 = subset.apply(
        lambda row: row['true_parent'] in [
            row['pred1_parent'],
            row['pred2_parent'],
            row['pred3_parent']
        ], axis=1
    ).mean()
    status = "✅" if t3 >= 0.5 else "⚠️ "
    print(f"   {status} {label:<33} {t1:>5.0%}   {t3:>5.0%}   {len(subset):>6,}")

# Save
df.to_csv('/home/arshad/Network-project/data/sample_with_mitre_matches.csv', index=False)
print("\n✅ Saved!")

📊 FINAL ACCURACY — Targeted Alert Text
   Top-1 Accuracy: 30.19%
   Top-3 Accuracy: 67.53%

Attack Type                          Top-1   Top-3    Count
----------------------------------------------------------
   ⚠️  Bot                                  0%      0%    1,953
   ⚠️  DDoS                                 0%      2%   128,016
   ✅ DoS GoldenEye                       72%     96%   10,286
   ✅ DoS Hulk                            12%     98%   172,849
   ✅ DoS Slowhttptest                    81%     94%    5,228
   ✅ DoS slowloris                       72%     75%    5,385
   ✅ FTP-Patator                         66%     67%    5,933
   ✅ Heartbleed                         100%    100%       11
   ⚠️  Infiltration                         0%      0%       36
   ✅ PortScan                            97%     99%   90,819
   ✅ SSH-Patator                          0%     92%    3,219
   ⚠️  Web Attack - Brute Force             0%      0%    1,470
   ⚠️  Web Attack - SQL Injection  

In [22]:
def generate_final_alert_text(row):
    dst_port  = int(row.get('Destination Port', 0))
    syn       = int(row.get('SYN Flag Count', 0))
    ack       = int(row.get('ACK Flag Count', 0))
    fin       = int(row.get('FIN Flag Count', 0))
    rst       = int(row.get('RST Flag Count', 0))
    psh       = int(row.get('PSH Flag Count', 0))
    fwd_pkts  = int(row.get('Total Fwd Packets', 0))
    bwd_pkts  = int(row.get('Total Backward Packets', 0))
    pkt_rate  = float(row.get('Flow Packets/s', 0))
    pkt_len   = float(row.get('Packet Length Mean', 0))
    byte_rate = float(row.get('Flow Bytes/s', 0))
    duration  = float(row.get('Flow Duration', 0)) / 1e6
    iat_mean  = float(row.get('Flow IAT Mean', 0))
    init_win  = int(row.get('Init_Win_bytes_forward', 0))
    down_up   = float(row.get('Down/Up Ratio', 0))
    flow_pkts = fwd_pkts + bwd_pkts
    fwd_bwd_ratio = fwd_pkts / (bwd_pkts + 1)

    port_context = {
        80: 'HTTP web server',      443: 'HTTPS web server',
        22: 'SSH remote access',    21:  'FTP file transfer server',
        23: 'Telnet service',       3389:'RDP remote desktop',
        53: 'DNS resolver',         3306:'MySQL database server',
        8080:'HTTP web proxy',      445: 'SMB file sharing',
    }
    port_desc = port_context.get(dst_port, f'port {dst_port}')

    flags = []
    if syn: flags.append('SYN')
    if ack: flags.append('ACK')
    if fin: flags.append('FIN')
    if rst: flags.append('RST')
    if psh: flags.append('PSH')
    flag_str = ', '.join(flags) if flags else 'no TCP flags'

    # ── 1. PORT SCAN (highest priority — tiny packets, high rate, few per flow)
    if pkt_len < 20 and flow_pkts <= 3 and pkt_rate > 1000:
        return (
            f"Network reconnaissance and port scanning activity probing {port_desc}. "
            f"Adversary enumerating open network services to discover attack surface. "
            f"Tiny {pkt_len:.1f} byte probe packets at {pkt_rate:.0f} packets/sec. "
            f"Single-packet flows with {flag_str} flags typical of automated port scanners. "
            f"Consistent with network service discovery and host enumeration."
        )

    # ── 2. BRUTE FORCE on auth ports (check BEFORE generic web/C2)
    if dst_port in [21, 22, 23, 3389, 5900]:
        service = {21:'FTP', 22:'SSH', 23:'Telnet',
                   3389:'RDP', 5900:'VNC'}.get(dst_port, 'remote')
        return (
            f"Credential brute force attack against {service} service on {port_desc}. "
            f"Adversary systematically attempting password guessing to gain "
            f"unauthorized access to {service} accounts. "
            f"{fwd_pkts} forward and {bwd_pkts} backward packets over {duration:.2f} seconds "
            f"at {pkt_rate:.1f} packets/sec with {flag_str} flags. "
            f"Repeated failed login attempts consistent with brute force "
            f"and password spraying credential access techniques."
        )

    # ── 3. DDOS — large packets, high volume, network layer (check before DoS)
    if pkt_len > 400 and pkt_rate > 50 and fwd_bwd_ratio > 2:
        return (
            f"Distributed denial of service network flood targeting {port_desc}. "
            f"High volume of {pkt_len:.0f} byte packets at {pkt_rate:.0f} packets/sec "
            f"consuming {byte_rate:.0f} bytes/sec of bandwidth. "
            f"Predominantly forward traffic ({fwd_pkts} sent vs {bwd_pkts} received) "
            f"with {flag_str} flags typical of volumetric DDoS flood. "
            f"Consistent with network denial of service and direct network flood attack."
        )

    # ── 4. DOS FLOOD — high rate unidirectional (after DDoS check)
    if pkt_rate > 200 and fwd_bwd_ratio > 3:
        return (
            f"Denial of service flood attack exhausting resources at {port_desc}. "
            f"Adversary sending {pkt_rate:.0f} packets/sec to overwhelm and crash target service. "
            f"{fwd_pkts} forward vs {bwd_pkts} backward packets with {flag_str} flags. "
            f"Byte rate {byte_rate:.0f} bytes/sec over {duration:.2f} seconds. "
            f"Consistent with endpoint denial of service and service exhaustion flood."
        )

    # ── 5. BOTNET C2 — periodic small bidirectional (BEFORE web attack check)
    # Key: high IAT (slow beaconing), small packets, truly bidirectional
    if iat_mean > 100000 and bwd_pkts > 0 and pkt_len < 200 and fwd_bwd_ratio < 3:
        return (
            f"Botnet command and control beaconing via {port_desc}. "
            f"Compromised host communicating with remote C2 server "
            f"using application layer protocol over {port_desc}. "
            f"Periodic {pkt_len:.1f} byte packets every {iat_mean/1000:.0f} milliseconds "
            f"bidirectional: {fwd_pkts} sent, {bwd_pkts} received. "
            f"Slow beaconing pattern with {flag_str} flags consistent with "
            f"bot malware maintaining persistent command and control channel."
        )

    # ── 6. EXPLOITATION — large payloads, long session, any port
    if pkt_len > 800 and duration > 10 and flow_pkts > 100:
        return (
            f"Remote service exploitation attempt targeting {port_desc}. "
            f"Large {pkt_len:.0f} byte malicious payloads over {duration:.1f} second session "
            f"exploiting software vulnerability for unauthorized access. "
            f"{fwd_pkts} forward and {bwd_pkts} backward packets with {flag_str} flags. "
            f"Extended high-payload session consistent with exploitation of "
            f"remote services and vulnerability exploitation for initial access."
        )

    # ── 7. LATERAL MOVEMENT / INFILTRATION — internal spreading behavior
    if bwd_pkts > fwd_pkts and duration > 5 and dst_port not in [80, 443, 8080]:
        return (
            f"Suspicious lateral movement activity via {port_desc}. "
            f"Adversary transferring tools or malware laterally across network. "
            f"More backward ({bwd_pkts}) than forward ({fwd_pkts}) packets "
            f"over {duration:.2f} seconds suggests data being pulled from compromised host. "
            f"Consistent with lateral tool transfer and internal network spreading."
        )

    # ── 8. WEB ATTACKS — SQL injection, XSS, web brute force
    if dst_port in [80, 443, 8080] and psh > 0:
        # Distinguish web brute force (many short requests) vs injection (fewer bigger)
        if pkt_len < 100 and flow_pkts > 5:
            return (
                f"Web application brute force attack against {port_desc}. "
                f"Adversary repeatedly submitting login requests attempting "
                f"credential brute force against web application. "
                f"{flow_pkts} short {pkt_len:.0f} byte HTTP requests over {duration:.2f} seconds. "
                f"High frequency web requests consistent with web login brute forcing "
                f"and exploitation of public-facing application authentication."
            )
        else:
            return (
                f"Web application attack targeting {port_desc} with malicious payloads. "
                f"Adversary injecting malicious content into web requests via "
                f"SQL injection or cross-site scripting XSS to exploit "
                f"public-facing application vulnerabilities. "
                f"{fwd_pkts} requests with {pkt_len:.0f} byte payloads over {duration:.2f} seconds "
                f"with {flag_str} flags. Consistent with exploit public-facing application."
            )

    # ── 9. DATA EXFILTRATION
    if down_up > 5 and byte_rate > 5000:
        return (
            f"Data exfiltration from {port_desc}. "
            f"Abnormally high outbound transfer at {byte_rate:.0f} bytes/sec "
            f"with down/up ratio {down_up:.1f} suggests automated data theft. "
            f"Consistent with exfiltration over alternative protocol or web service."
        )

    # ── 10. DEFAULT
    return (
        f"Network flow targeting {port_desc} with {flag_str} flags. "
        f"{fwd_pkts} forward and {bwd_pkts} backward packets "
        f"at {pkt_rate:.1f} packets/sec, {pkt_len:.1f} byte mean packet size, "
        f"{duration:.2f} second duration, {byte_rate:.1f} bytes/sec."
    )

print("✅ Final alert generator defined!")
print("\n📝 Previewing all attack types:")
print("=" * 70)
for label in sorted(df['Label'].unique()):
    row = df[df['Label'] == label].iloc[0]
    text = generate_final_alert_text(row)
    print(f"\n🏷️  {label}")
    print(f"   {text[:220]}")

✅ Final alert generator defined!

📝 Previewing all attack types:

🏷️  BENIGN
   Network flow targeting HTTP web server with ACK flags. 2 forward and 0 backward packets at 82.2 packets/sec, 6.0 byte mean packet size, 0.02 second duration, 493.3 bytes/sec.

🏷️  Bot
   Web application brute force attack against HTTP web proxy. Adversary repeatedly submitting login requests attempting credential brute force against web application. 7 short 43 byte HTTP requests over 0.07 seconds. High f

🏷️  DDoS
   Web application attack targeting HTTP web server with malicious payloads. Adversary injecting malicious content into web requests via SQL injection or cross-site scripting XSS to exploit public-facing application vulnera

🏷️  DoS GoldenEye
   Network flow targeting HTTP web server with ACK flags. 2 forward and 0 backward packets at 0.3 packets/sec, 0.0 byte mean packet size, 7.66 second duration, 0.0 bytes/sec.

🏷️  DoS Hulk
   Network flow targeting HTTP web server with FIN flags. 6 forward an

In [23]:
from tqdm import tqdm
tqdm.pandas()

# Generate
print("⏳ Generating final alert texts...")
df['alert_text'] = df.progress_apply(generate_final_alert_text, axis=1)

# Embed
print("\n⏳ Embedding...")
alert_emb_final = model.encode(
    df['alert_text'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)
faiss.normalize_L2(alert_emb_final)

# Search
sims, idxs = index.search(alert_emb_final, 3)
for rank in range(3):
    df[f'pred_technique_{rank+1}']  = [techniques[i]['technique_id']   for i in idxs[:, rank]]
    df[f'pred_tech_name_{rank+1}']  = [techniques[i]['technique_name'] for i in idxs[:, rank]]
    df[f'pred_tactic_{rank+1}']     = [techniques[i]['tactics']        for i in idxs[:, rank]]
    df[f'pred_confidence_{rank+1}'] = sims[:, rank].round(4)

# Evaluate
attack_df = df[df['Label'] != 'BENIGN'].copy()
attack_df['true_parent']  = attack_df['MITRE_Technique'].apply(get_parent_id)
attack_df['pred1_parent'] = attack_df['pred_technique_1'].apply(get_parent_id)
attack_df['pred2_parent'] = attack_df['pred_technique_2'].apply(get_parent_id)
attack_df['pred3_parent'] = attack_df['pred_technique_3'].apply(get_parent_id)

top1 = (attack_df['true_parent'] == attack_df['pred1_parent']).mean()
top3 = attack_df.apply(
    lambda row: row['true_parent'] in [
        row['pred1_parent'],
        row['pred2_parent'],
        row['pred3_parent']
    ], axis=1
).mean()

print("\n📊 FINAL ACCURACY")
print("=" * 55)
print(f"   Top-1 Accuracy: {top1:.2%}")
print(f"   Top-3 Accuracy: {top3:.2%}")
print()
print(f"{'Attack Type':<35} {'Top-1':>6}  {'Top-3':>6}  {'Count':>7}")
print("-" * 58)
for label in sorted(attack_df['Label'].unique()):
    subset = attack_df[attack_df['Label'] == label]
    t1 = (subset['true_parent'] == subset['pred1_parent']).mean()
    t3 = subset.apply(
        lambda row: row['true_parent'] in [
            row['pred1_parent'],
            row['pred2_parent'],
            row['pred3_parent']
        ], axis=1
    ).mean()
    status = "✅" if t3 >= 0.5 else "⚠️ "
    print(f"   {status} {label:<33} {t1:>5.0%}   {t3:>5.0%}   {len(subset):>6,}")

# Save
df.to_csv('/home/arshad/Network-project/data/sample_with_mitre_matches.csv', index=False)
print("\n✅ Saved!")

⏳ Generating final alert texts...


100%|████████████████████████████████| 435878/435878 [00:07<00:00, 57186.04it/s]



⏳ Embedding...


Batches:   0%|          | 0/1703 [00:00<?, ?it/s]


📊 FINAL ACCURACY
   Top-1 Accuracy: 28.84%
   Top-3 Accuracy: 67.24%

Attack Type                          Top-1   Top-3    Count
----------------------------------------------------------
   ⚠️  Bot                                  0%     26%    1,953
   ⚠️  DDoS                                 0%      0%   128,016
   ✅ DoS GoldenEye                       68%     97%   10,286
   ✅ DoS Hulk                            11%     99%   172,849
   ✅ DoS Slowhttptest                    61%     76%    5,228
   ✅ DoS slowloris                       34%     67%    5,385
   ✅ FTP-Patator                         68%     68%    5,933
   ✅ Heartbleed                         100%    100%       11
   ⚠️  Infiltration                         0%      0%       36
   ✅ PortScan                            97%     99%   90,819
   ✅ SSH-Patator                          0%     93%    3,219
   ⚠️  Web Attack - Brute Force             0%      0%    1,470
   ⚠️  Web Attack - SQL Injection           0%      0%  

In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Features for classification
feature_cols = [
    'Destination Port', 'Flow Duration',
    'Total Fwd Packets', 'Total Backward Packets',
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Packet Length Mean', 'Packet Length Std',
    'Flow Bytes/s', 'Flow Packets/s',
    'Fwd Packets/s', 'Bwd Packets/s',
    'Flow IAT Mean', 'Flow IAT Std',
    'SYN Flag Count', 'ACK Flag Count',
    'FIN Flag Count', 'RST Flag Count',
    'PSH Flag Count', 'URG Flag Count',
    'Init_Win_bytes_forward', 'Init_Win_bytes_backward',
    'Active Mean', 'Idle Mean',
    'Down/Up Ratio', 'Average Packet Size'
]

# Use tactic as target (coarser = easier to learn)
tactic_map = {
    'BENIGN':                    'benign',
    'DoS Hulk':                  'impact',
    'DoS GoldenEye':             'impact',
    'DoS slowloris':             'impact',
    'DoS Slowhttptest':          'impact',
    'DDoS':                      'impact',
    'PortScan':                  'discovery',
    'FTP-Patator':               'credential-access',
    'SSH-Patator':               'credential-access',
    'Bot':                       'command-and-control',
    'Web Attack - Brute Force':  'initial-access',
    'Web Attack - XSS':          'initial-access',
    'Web Attack - SQL Injection':'initial-access',
    'Heartbleed':                'initial-access',
    'Infiltration':              'lateral-movement',
}

df['tactic_label'] = df['Label'].map(tactic_map)

# Prepare data
X = df[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
y = df['tactic_label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"✅ Training set: {len(X_train):,} samples")
print(f"✅ Test set:     {len(X_test):,} samples")
print(f"✅ Tactics:      {le.classes_}")

✅ Training set: 348,702 samples
✅ Test set:     87,176 samples
✅ Tactics:      ['benign' 'command-and-control' 'credential-access' 'discovery' 'impact'
 'initial-access' 'lateral-movement']


In [25]:
print("⏳ Training Random Forest classifier...")

clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    n_jobs=-1,         # use all CPU cores
    random_state=42,
    class_weight='balanced'
)

clf.fit(X_train, y_train)
print("✅ Training complete!")

# Evaluate
y_pred = clf.predict(X_test)
print("\n📊 TACTIC CLASSIFICATION REPORT:")
print("=" * 60)
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    digits=3
))

⏳ Training Random Forest classifier...
✅ Training complete!

📊 TACTIC CLASSIFICATION REPORT:
                     precision    recall  f1-score   support

             benign      0.987     0.986     0.987      2000
command-and-control      0.965     1.000     0.982       391
  credential-access      0.998     0.999     0.999      1830
          discovery      1.000     0.999     1.000     18164
             impact      1.000     1.000     1.000     64353
     initial-access      0.975     0.995     0.985       431
   lateral-movement      1.000     0.714     0.833         7

           accuracy                          0.999     87176
          macro avg      0.989     0.956     0.969     87176
       weighted avg      0.999     0.999     0.999     87176



In [26]:
import pickle

# Save classifier
with open('/home/arshad/Network-project/data/tactic_classifier.pkl', 'wb') as f:
    pickle.dump({'model': clf, 'encoder': le, 'features': feature_cols}, f)
print("✅ Classifier saved!")

def hybrid_mitre_match(row, model_emb, index, techniques, clf, le, feature_cols, top_k=3):
    """
    Hybrid approach:
    1. Use ML classifier to predict MITRE tactic
    2. Use embedding to find best technique WITHIN that tactic
    3. Fall back to global embedding if tactic match fails
    """
    # Step 1: ML tactic prediction
    features = pd.DataFrame([row[feature_cols]])
    features = features.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    tactic_probs  = clf.predict_proba(features)[0]
    tactic_idx    = tactic_probs.argmax()
    pred_tactic   = le.classes_[tactic_idx]
    tactic_conf   = tactic_probs[tactic_idx]

    # Step 2: Embed alert text
    alert_emb = model_emb.encode([row['alert_text']], convert_to_numpy=True)
    faiss.normalize_L2(alert_emb)

    # Step 3: Filter techniques by predicted tactic
    tactic_indices = [
        i for i, t in enumerate(techniques)
        if pred_tactic in t['tactics']
    ]

    results = []
    if len(tactic_indices) >= top_k:
        # Search within tactic-filtered techniques only
        tactic_embeddings_sub = technique_embeddings[tactic_indices]
        sims = (alert_emb @ tactic_embeddings_sub.T)[0]
        top_local = np.argsort(sims)[::-1][:top_k]

        for rank_i in top_local:
            global_i = tactic_indices[rank_i]
            t = techniques[global_i]
            results.append({
                'technique_id':   t['technique_id'],
                'technique_name': t['technique_name'],
                'tactics':        t['tactics'],
                'confidence':     round(float(sims[rank_i]), 4),
                'tactic_source':  'ml_classifier',
                'pred_tactic':    pred_tactic,
                'tactic_conf':    round(float(tactic_conf), 4)
            })
    else:
        # Fall back to global search
        sims, idxs = index.search(alert_emb, top_k)
        for sim, idx in zip(sims[0], idxs[0]):
            t = techniques[idx]
            results.append({
                'technique_id':   t['technique_id'],
                'technique_name': t['technique_name'],
                'tactics':        t['tactics'],
                'confidence':     round(float(sim), 4),
                'tactic_source':  'global_fallback',
                'pred_tactic':    pred_tactic,
                'tactic_conf':    round(float(tactic_conf), 4)
            })

    return results

print("✅ Hybrid matching function defined!")

✅ Classifier saved!
✅ Hybrid matching function defined!


In [1]:
from tqdm import tqdm

print("⏳ Applying hybrid matching to all alerts...")
print("   (Using vectorized ML predictions for speed)\n")

# Batch ML predictions (fast)
X_all = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
tactic_preds    = clf.predict(X_all)
tactic_probs    = clf.predict_proba(X_all)
pred_tactics    = le.inverse_transform(tactic_preds)
pred_tactic_conf = tactic_probs.max(axis=1)

df['ml_pred_tactic']     = pred_tactics
df['ml_tactic_conf']     = pred_tactic_conf.round(4)

# Load technique embeddings
technique_embeddings_arr = np.load(
    '/home/arshad/Network-project/data/mitre_embeddings.npy'
)

# For each alert, find best technique within ML-predicted tactic
print("⏳ Running tactic-filtered embedding search...")

pred1_ids, pred1_names, pred1_tactics_out, pred1_confs = [], [], [], []
pred2_ids, pred2_names, pred2_tactics_out, pred2_confs = [], [], [], []
pred3_ids, pred3_names, pred3_tactics_out, pred3_confs = [], [], [], []

# Batch embed alert texts (already done — reload)
alert_emb_arr = np.load('/home/arshad/Network-project/data/alert_embeddings.npy')

# Re-embed with final alert texts
print("⏳ Re-embedding final alert texts...")
alert_emb_final = model.encode(
    df['alert_text'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)
faiss.normalize_L2(alert_emb_final)

for i in tqdm(range(len(df)), desc="Hybrid matching"):
    pred_tactic = pred_tactics[i]

    # Get indices of techniques matching predicted tactic
    tactic_indices = [
        j for j, t in enumerate(techniques)
        if pred_tactic in t['tactics']
    ]

    if len(tactic_indices) >= 3:
        sub_embs = technique_embeddings_arr[tactic_indices]
        sims     = (alert_emb_final[i:i+1] @ sub_embs.T)[0]
        top3_local = np.argsort(sims)[::-1][:3]
        top3_global = [tactic_indices[k] for k in top3_local]
        top3_sims   = sims[top3_local]
    else:
        # fallback: global search
        s, idx = index.search(alert_emb_final[i:i+1], 3)
        top3_global = idx[0].tolist()
        top3_sims   = s[0]

    for rank_out, (gidx, sim) in enumerate(zip(top3_global, top3_sims)):
        t = techniques[gidx]
        if rank_out == 0:
            pred1_ids.append(t['technique_id']); pred1_names.append(t['technique_name'])
            pred1_tactics_out.append(t['tactics']); pred1_confs.append(round(float(sim),4))
        elif rank_out == 1:
            pred2_ids.append(t['technique_id']); pred2_names.append(t['technique_name'])
            pred2_tactics_out.append(t['tactics']); pred2_confs.append(round(float(sim),4))
        else:
            pred3_ids.append(t['technique_id']); pred3_names.append(t['technique_name'])
            pred3_tactics_out.append(t['tactics']); pred3_confs.append(round(float(sim),4))

df['pred_technique_1'] = pred1_ids;  df['pred_tech_name_1'] = pred1_names
df['pred_tactic_1']    = pred1_tactics_out; df['pred_confidence_1'] = pred1_confs
df['pred_technique_2'] = pred2_ids;  df['pred_tech_name_2'] = pred2_names
df['pred_tactic_2']    = pred2_tactics_out; df['pred_confidence_2'] = pred2_confs
df['pred_technique_3'] = pred3_ids;  df['pred_tech_name_3'] = pred3_names
df['pred_tactic_3']    = pred3_tactics_out; df['pred_confidence_3'] = pred3_confs

print("\n✅ Hybrid matching complete!")

⏳ Applying hybrid matching to all alerts...
   (Using vectorized ML predictions for speed)



NameError: name 'df' is not defined